# MoneyPrinterTurbo → Qwen-Image → Remotion

Renderiza un vídeo vertical a partir del paquete ZIP exportado desde **De tendencia a vídeo**. Qwen genera imágenes originales; Remotion compone escenas, clips propios y una locución opcional.

**Antes de empezar:** crea `Mi unidad/MoneyPrinterTurbo/mpt-container/` y sube solo las carpetas `remotion/` y `colab/` del proyecto; copia el ZIP descargado desde la WebUI a `remotion-inbox/`. Selecciona una GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Los clips solo se copian a Drive cuando tú subes el ZIP. No pongas `config.toml`, claves API ni credenciales en Drive o en el ZIP.

El render de Remotion usa Chromium/FFmpeg y suele depender de CPU/RAM; la GPU se utiliza principalmente para generar las imágenes Qwen. Colab no garantiza un tipo concreto de GPU ni memoria suficiente. Qwen-Image-2.1 publica una licencia **Qwen Research License** (no Apache); revisa sus términos antes de uso comercial. Remotion también tiene una licencia propia.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/MoneyPrinterTurbo')
PROJECT_DIR = DRIVE_ROOT / 'mpt-container'
sys.path.insert(0, str(PROJECT_DIR))
INBOX_DIR = DRIVE_ROOT / 'remotion-inbox'
OUTPUT_DIR = DRIVE_ROOT / 'renders'
INBOX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not (PROJECT_DIR / 'remotion' / 'package.json').is_file():
    raise FileNotFoundError(f'No encuentro el proyecto Remotion: {PROJECT_DIR}. Sube la carpeta mpt-container a Drive.')
sys.path.insert(0, str(PROJECT_DIR / 'colab'))
print('Proyecto:', PROJECT_DIR)
print('Entrada ZIP:', INBOX_DIR)
print('Vídeos terminados:', OUTPUT_DIR)


## Instalar dependencias

El código de la app usa los paquetes publicados de Remotion, no clona ni instala el monorepo completo del framework. La descarga del modelo Qwen es grande y queda en el almacenamiento temporal de Colab; al reiniciar el runtime puede ser necesario descargarlo otra vez.


In [ ]:
%pip install --upgrade 'transformers>=5.17' accelerate pillow 'git+https://github.com/huggingface/diffusers'

import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run(['node', '--version'], check=True)
subprocess.run(['npm', '--version'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.DEVNULL)


## Renderizar

Coloca el ZIP descargado en `Mi unidad/MoneyPrinterTurbo/remotion-inbox`. Si hay varios, se procesará el más reciente. Cada salida guarda el MP4, las imágenes generadas y los manifiestos en su propia carpeta bajo `renders/`.


In [ ]:
import torch
from colab_runner import IMAGE_MODEL_ID, render_bundle

if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
bundles = sorted(INBOX_DIR.glob('*.zip'), key=lambda path: path.stat().st_mtime_ns)
if not bundles:
    raise FileNotFoundError(f'Sube remotion-project.zip a {INBOX_DIR}')
BUNDLE_PATH = bundles[-1]
print(f'Paquete: {BUNDLE_PATH.name}')
print(f'Modelo de imagen: {IMAGE_MODEL_ID}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
rendered_video = render_bundle(
    bundle_path=BUNDLE_PATH,
    project_template=PROJECT_DIR / 'remotion',
    output_dir=OUTPUT_DIR,
    generate_images=True,
    model_id=IMAGE_MODEL_ID,
)
print('Vídeo guardado en Drive:', rendered_video)


In [ ]:
from IPython.display import Video, display
display(Video(str(rendered_video), embed=False))
